# 02 · Объединение и признаки

Ноутбук собирает из очищенных таблиц две аналитические таблицы разного зерна (L5: «одна строка = что?»):

- **часовая** `hourly.csv` — одна строка = один час на сетке станции; для H3 (суточный цикл) и H4 (CAMS и станция);
- **суточная** `daily.csv` — одна строка = одни местные сутки Алматы; для H1, H2, нормы ВОЗ и модели.

Входы: `pm25_clean.csv`, `weather_clean.csv`, `cams_clean.csv` из `data/processed/`. Выходы: `hourly.csv` и `daily.csv` там же.

Порядок: читаем и проверяем → выравниваем осадки → объединяем → признаки времени → запись `hourly.csv` → суточная таблица (часть 2). Запускать сверху вниз.

In [1]:
import sys

import pandas as pd

sys.path.append("..")
from src import clean, collect, features

PM25_CLEAN_PATH = "../data/processed/pm25_clean.csv"
WEATHER_CLEAN_PATH = "../data/processed/weather_clean.csv"
CAMS_CLEAN_PATH = "../data/processed/cams_clean.csv"
HOURLY_PATH = "../data/processed/hourly.csv"
DAILY_PATH = "../data/processed/daily.csv"

WEATHER_COLUMNS = list(collect.WEATHER_COLUMNS.values())
CAMS_COLUMNS = list(collect.CAMS_COLUMNS.values())

## Шаг 1. Читаем очищенные таблицы и проверяем контракты

Сначала убеждаемся, что входы соответствуют контрактам этапа 2. Отдельно проверяем границы: погода должна начинаться не позже станции и заканчиваться минимум через час после её последней метки — это нужно для сдвига осадков на следующем шаге.

In [2]:
pm25 = clean.read_clean_csv(PM25_CLEAN_PATH)
weather = clean.read_clean_csv(WEATHER_CLEAN_PATH)
cams = clean.read_clean_csv(CAMS_CLEAN_PATH)

clean.check_clean_pm25(pm25)
clean.check_clean_hourly(weather, WEATHER_COLUMNS)
clean.check_clean_hourly(cams, CAMS_COLUMNS)

print("pm25:", pm25.shape, "| weather:", weather.shape, "| cams:", cams.shape)

# погода покрывает станцию и ещё час после её последней метки
assert weather["time_utc"].min() <= pm25["time_utc"].min()
assert weather["time_utc"].max() >= pm25["time_utc"].max() + pd.Timedelta(hours=1)

print("станция:", pm25["time_utc"].min(), "...", pm25["time_utc"].max())
print("погода: ", weather["time_utc"].min(), "...", weather["time_utc"].max())
print("CAMS начинается:", cams["time_utc"].min())

pm25: (49075, 4) | weather: (49104, 7) | cams: (28776, 2)
станция: 2020-04-09 16:00:00+00:00 ... 2025-11-14 10:00:00+00:00
погода:  2020-04-09 00:00:00+00:00 ... 2025-11-14 23:00:00+00:00
CAMS начинается: 2022-08-04 00:00:00+00:00


## Шаг 2. Осадки: тот же час, что у PM2.5

Метка времени у станции — начало часа: PM2.5 в строке T — среднее за [T, T+1 ч). У Open-Meteo `temperature_2m`, влажность, давление и ветер — мгновенные значения в момент T, а `precipitation` — «сумма за предыдущий час» (по документации Open-Meteo: *Preceding hour sum*). Значит, осадки в метке T относятся к часу [T−1 ч, T).

Чтобы осадки и PM2.5 в одной строке описывали один и тот же час, берём осадки из следующей строки (`shift(-1)`) — на полной часовой сетке это следующий час. Без сдвига суточная сумма попала бы на час мимо суток (решение — ADR-014).

In [3]:
weather_aligned = features.align_precip_to_hour_start(weather)

# первые часы рядом: до и после сдвига
compare = pd.DataFrame(
    {
        "time_utc": weather["time_utc"],
        "precip_before": weather["precip"],
        "precip_after": weather_aligned["precip"],
    }
)
print(compare[compare["precip_before"] > 0].head(5))

# сумма осадков на часах станции: сдвиг не должен менять общий объём
station_hours = pm25["time_utc"]
before = weather[weather["time_utc"].isin(station_hours)]["precip"].sum()
after = weather_aligned[weather_aligned["time_utc"].isin(station_hours)]["precip"].sum()
print("сумма осадков на часах станции, до:", round(before, 1), "| после:", round(after, 1))
print("NaN в precip после сдвига (всего):", weather_aligned["precip"].isna().sum())

# суточные суммы по местным датам: сколько суток изменилось
dates = features.local_dates(weather["time_utc"])
sum_before = weather.groupby(dates)["precip"].sum()
sum_after = weather_aligned.groupby(dates)["precip"].sum()
hours_in_day = weather.groupby(dates)["time_utc"].size()
full_days = hours_in_day[hours_in_day >= 24].index
diff = (sum_after[full_days] - sum_before[full_days]).abs()
print("полных суток с изменившейся суточной суммой:", (diff > 1e-9).sum())
print("максимум разницы суточной суммы, мм:", round(diff.max(), 1))

                   time_utc  precip_before  precip_after
0 2020-04-09 00:00:00+00:00            0.5           0.5
1 2020-04-09 01:00:00+00:00            0.5           0.3
2 2020-04-09 02:00:00+00:00            0.3           0.2
3 2020-04-09 03:00:00+00:00            0.2           0.4
4 2020-04-09 04:00:00+00:00            0.4           0.6
сумма осадков на часах станции, до: 2925.2 | после: 2925.2
NaN в precip после сдвига (всего): 1
полных суток с изменившейся суточной суммой: 303
максимум разницы суточной суммы, мм: 7.4


## Шаг 3. Объединение по `time_utc`

Строки задаёт станция: `how="left"` от таблицы PM2.5, поэтому сетка станции не меняется (ADR-012). `validate="one_to_one"` — защита от тихого размножения строк, если в погоде или CAMS вдруг окажется повторяющийся час; после каждого `merge` функция проверяет число строк.

CAMS в Open-Meteo начинается с 2022-08-04 — раньше значений нет. Это период источника, а не пропуск, поэтому `cams_pm25` до этой даты остаётся пустым и мы его не заполняем.

In [4]:
merged = features.merge_sources(pm25, weather_aligned, cams)

print("строк: станция", len(pm25), "-> после объединения", len(merged))
print(merged.isna().sum())

# пустой CAMS ровно там, где час раньше первой метки источника
first_cams = cams["time_utc"].min()
hours_before_cams = (merged["time_utc"] < first_cams).sum()
print("часов станции раньше начала CAMS:", hours_before_cams)
assert merged["cams_pm25"].isna().sum() == hours_before_cams

строк: станция 49075 -> после объединения 49075
time_utc                 0
pm25                 10235
pm25_missing             0
pm25_interpolated        0
temp_c                   0
humidity                 0
precip                   0
pressure                 0
wind_speed               0
wind_dir                 0
cams_pm25            20312
dtype: int64
часов станции раньше начала CAMS: 20312


## Шаг 4. Признаки времени по местному времени

Ключ таблицы остаётся `time_utc`, а признаки считаем по местному времени Алматы: суточный цикл и сезон отопления определяются жизнью города, а не UTC. 18:00 UTC — это уже полночь в Алматы и следующий день. 1 марта 2024 в Алматы сменился пояс (UTC+6 → UTC+5), поэтому местные сутки 29.02.2024 длятся 25 часов.

Определения:
- `season` — метеорологические сезоны по месяцу: зима — дек–фев, весна — мар–май, лето — июн–авг, осень — сен–ноя;
- `heating_season` — с 15 октября по 15 апреля, обе границы включительно (ориентир; реальные даты каждый год свои);
- `is_weekend` — суббота и воскресенье;
- `is_calm` — штиль: ветер строго меньше 1 м/с.

In [5]:
hourly = features.make_hourly(merged)
print(hourly.head())

print(hourly["season"].value_counts())
print(hourly["heating_season"].value_counts())
print("доля часов со штилем:", round(hourly["is_calm"].mean(), 4))

# смена пояса: местный час 23 встречается дважды
around_change = hourly[
    (hourly["time_utc"] >= pd.Timestamp("2024-02-29 17:00", tz="UTC"))
    & (hourly["time_utc"] <= pd.Timestamp("2024-02-29 19:00", tz="UTC"))
]
print(around_change[["time_utc", "hour", "month"]])

                   time_utc  pm25  pm25_missing  pm25_interpolated  temp_c  \
0 2020-04-09 16:00:00+00:00   6.0         False              False    -2.4   
1 2020-04-09 17:00:00+00:00  10.0         False              False    -2.2   
2 2020-04-09 18:00:00+00:00   8.0         False              False    -4.4   
3 2020-04-09 19:00:00+00:00   6.0         False              False    -5.0   
4 2020-04-09 20:00:00+00:00   4.0         False              False    -5.9   

   humidity  precip  pressure  wind_speed  wind_dir  cams_pm25  hour  \
0        84     0.0    1039.4        1.88       115        NaN    22   
1        72     0.0    1039.0        1.70       130        NaN    23   
2        68     0.0    1038.7        2.14       127        NaN     0   
3        71     0.0    1038.0        2.06       129        NaN     1   
4        73     0.0    1037.5        1.98       131        NaN     2   

   dayofweek  month  is_weekend  season  heating_season  is_calm  
0          3      4       False

## Шаг 5. Проверки и запись `hourly.csv`

Контракт `check_hourly` вызываем до записи и после чтения. Формат времени тот же, что у очищенных файлов (`+0000`). Файл перезаписывается при каждом запуске. После чтения сравниваем таблицу с исходной: типы `hour`, `dayofweek` и `month` после чтения CSV становятся `int64` (до записи — `int32`), поэтому сравниваем значения без учёта типа.

In [6]:
features.check_hourly(hourly)
clean.save_clean_csv(hourly, HOURLY_PATH)

hourly_read = clean.read_clean_csv(HOURLY_PATH)
features.check_hourly(hourly_read)
pd.testing.assert_frame_equal(hourly, hourly_read, check_dtype=False)

print("hourly:", hourly.shape, "| после чтения:", hourly_read.shape)

with open(HOURLY_PATH, encoding="utf-8") as file:
    print(file.readline().strip())
    print(file.readline().strip())

hourly: (49075, 18) | после чтения: (49075, 18)
time_utc,pm25,pm25_missing,pm25_interpolated,temp_c,humidity,precip,pressure,wind_speed,wind_dir,cams_pm25,hour,dayofweek,month,is_weekend,season,heating_season,is_calm
2020-04-09T16:00:00+0000,6.0,False,False,-2.4,84,0.0,1039.4,1.88,115,,22,3,4,False,spring,True,False
